Vamos a tantear diferentes modelos y configuraciones antes de parametrizar el definitivo. Gracias a la libreria lazypredict hara un tanteo en los principales modelos de regresion y clasificacion.



In [1]:
import pandas as pd
import numpy as np, random
random.seed(42)

In [2]:
df2 = pd.read_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/models/presplit.csv')

In [3]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12800 entries, 0 to 12799
Data columns (total 34 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   cntry                           12800 non-null  object 
 1   pplfair                         12800 non-null  int64  
 2   pplhlp                          12800 non-null  int64  
 3   ppltrst                         12800 non-null  int64  
 4   lrscale                         12800 non-null  int64  
 5   polintr                         12800 non-null  int64  
 6   stfdem                          12800 non-null  int64  
 7   stfeco                          12800 non-null  int64  
 8   stfgov                          12800 non-null  int64  
 9   trstep                          12800 non-null  int64  
 10  trstlgl                         12800 non-null  int64  
 11  trstplc                         12800 non-null  int64  
 12  trstplt                         

In [14]:
import pandas as pd

def redondear_columnas(df, columnas):

    for columna in columnas:
        if columna in df.columns:
            df[columna] = df[columna].apply(lambda x: round(x, 1) if pd.notnull(x) else x)
        else:
            print(f"La columna '{columna}' no existe en el DataFrame.")
    return df

columnas_a_redondear = ['confianza_promedio', 'satisf_media', 'ppl']

# Aplicar el redondeo a las columnas especificadas
df2= redondear_columnas(df2, columnas_a_redondear)

# Verificar los valores redondeados
print(df2.head(4))

   pplfair  pplhlp  ppltrst  lrscale  polintr  stfdem  stfeco  stfgov  trstep  \
0        5       2        8        8        2       9       8       8       8   
1        5       5        5        0        4       5       6       5       5   
2        6       8        6        6        3       6       8       7       4   
3        7       7       10        5        3       3       5       2       2   

   trstlgl  ...  happyfc  confianza_promedio  confianza_promedio_factorizada  \
0        0  ...        2                 5.0                               2   
1        5  ...        1                 4.7                               2   
2        4  ...        1                 4.9                               2   
3        5  ...        2                 3.6                               1   

   satisf_media  satisf_media_factorizada  pintfc  ppl  ppl_fc  lrscale_fc  \
0           8.3                         4       1  5.0       2           3   
1           5.3                      

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
import os


def bestclass(df, target_column, feature_range, exclude_columns=None):


    results = {}
    X = df.drop(target_column, axis=1)
    y = df[target_column]

    if exclude_columns:
        X = X.drop(exclude_columns, axis=1)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    classifiers = {
        "Random Forest": RandomForestClassifier(random_state=42),
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
        "LightGBM": LGBMClassifier(random_state=42),
        "SVM": SVC(probability=True, random_state=42)
    }

    best_model = None
    best_auc = -float('inf')
    best_k = 0
    best_features = []
    best_classifier_name = ""

    for k in range(feature_range[0], feature_range[1] + 1):
        for classifier_name, classifier in classifiers.items():
            selector = SelectKBest(score_func=f_classif, k=k)
            X_train_selected = selector.fit_transform(X_train, y_train)
            X_test_selected = selector.transform(X_test)

            classifier.fit(X_train_selected, y_train)
            y_pred_prob = classifier.predict_proba(X_test_selected)

            auc = roc_auc_score(y_test, y_pred_prob, multi_class='ovr')

            if auc > best_auc:
                best_auc = auc
                best_model = classifier
                best_k = k
                best_features = X.columns[selector.get_support()].tolist()
                best_classifier_name = classifier_name

    output_dir = 'C:/Users/Josue/4GA.Datascience/4GA.DataScience/App'
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(output_dir, 'bestclassifier_iterative_corrected.txt'), 'w') as f:
        f.write(f"Mejor modelo: {best_classifier_name}\n")
        f.write(f"AUC: {best_auc}\n")
        f.write(f"Número de características: {best_k}\n")
        f.write(f"Características: {best_features}\n")

    return best_model, best_k, best_features, best_classifier_name


exclude_cols = ['lrscale', 'lrscale_2fc']  
best_model, best_k, best_features, best_classifier_name = bestclass(df2, 'lrscale_fc', (10, 18), exclude_columns=exclude_cols)

print(f"Mejor modelo: {best_classifier_name}")
print(f"Número de características: {best_k}")
print(f"Características: {best_features}")

: 